
# ♻️ E-Waste Detection + Classification (YOLO11) — Kaggle Notebook

**Pipeline overview**  
- Install & imports (Ultralytics 8.3.32 compatible)  
- Dataset setup (copy from `/kaggle/input/e-waste` → `/kaggle/working/e-waste`, rewrite `data.yaml` to be writable + cacheable)  
- Visualization (robust to extra values in Roboflow labels)  
- **Detector** training (YOLO11n, low-VRAM T4 safe) + validation → CSV  
- GT-crop generation (robust parser) to build classification dataset  
- Classifier split **sync** (make `val/` classes match `train/` so metrics are valid)  
- **Classifier** training + validation → CSV (+ confusion matrix)  
- One-cell **summary report**  
- (Optional) End-to-end inference with annotated outputs

> Notes:  
> • Keeps memory low for Tesla T4.  
> • Patches Ultralytics 8.3.32 Ray-Tune callback bug **without** needing to restart.  
> • Classification uses **directory path** (not YAML).  


## 1) Install & imports

In [ ]:

!pip -q install ultralytics==8.3.32 opencv-python matplotlib pandas

import os, glob, random, shutil, yaml, gc, torch, cv2
import numpy as np, matplotlib.pyplot as plt, matplotlib.patches as patches
from pathlib import Path
import pandas as pd
from ultralytics import YOLO

# Reproducibility + CUDA memory hygiene
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
gc.collect(); torch.cuda.empty_cache()

print('Setup OK. Torch:', torch.__version__)


In [7]:
# Run once per session (safe to re-run)
import types, importlib, sys
try:
    ray = importlib.import_module("ray")
except Exception:
    ray = types.SimpleNamespace(); sys.modules["ray"] = ray
if not hasattr(ray, "train"): ray.train = types.SimpleNamespace()
if not hasattr(ray.train, "_internal"): ray.train._internal = types.SimpleNamespace()
ray.train._internal.session = types.SimpleNamespace(_get_session=lambda: None)

from ultralytics.utils import callbacks as _ulx_cb
for k, v in list(_ulx_cb.default_callbacks.items()):
    if isinstance(v, list):
        _ulx_cb.default_callbacks[k] = [cb for cb in v if "raytune" not in getattr(cb, "__module__", "")]
    elif callable(v) and "raytune" in getattr(v, "__module__", ""):
        _ulx_cb.default_callbacks.pop(k, None)
print("Patched Ray callback ✅")


Patched Ray callback ✅
The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


## 2) Dataset setup (copy to `/kaggle/working` for caching)

In [ ]:

SRC = Path("/kaggle/input/e-waste")
assert (SRC/"data.yaml").exists(), "Expected /kaggle/input/e-waste/data.yaml"

DST = Path("/kaggle/working/e-waste")
if not DST.exists():
    shutil.copytree(SRC, DST)

# rewrite data.yaml to absolute paths under /kaggle/working
with open(SRC/"data.yaml") as f:
    cfg = yaml.safe_load(f)

cfg["train"] = str(DST/"train/images")
cfg["val"]   = str(DST/"valid/images")
if (DST/"test/images").exists():
    cfg["test"] = str(DST/"test/images")

DATA_YAML = Path("/kaggle/working/data.yaml")
with open(DATA_YAML, "w") as f:
    yaml.safe_dump(cfg, f)

names = cfg["names"]
print("Classes:", len(names))
print("Using data.yaml:", DATA_YAML)

train_dir = DST/"train/images"
val_dir   = DST/"valid/images"
test_dir  = DST/"test/images"
def count_imgs(p): 
    return len(list(p.glob("*.jpg"))) + len(list(p.glob("*.png"))) + len(list(p.glob("*.jpeg")))

print("Train images:", count_imgs(train_dir))
print("Val images:  ", count_imgs(val_dir))
print("Test images: ", count_imgs(test_dir) if test_dir.exists() else 0)


## 3) Visual sanity check (robust to extra label values)

In [ ]:

def show_one(img_path):
    img = cv2.imread(str(img_path)); img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lbl = Path(str(img_path).replace("/images/","/labels/")).with_suffix(".txt")
    fig, ax = plt.subplots(1,1,figsize=(6,6)); ax.imshow(img)
    if lbl.exists():
        for ln in lbl.read_text().splitlines():
            if not ln.strip(): continue
            vals = ln.split()
            if len(vals) < 5: continue
            c, x, y, bw, bh = map(float, vals[:5])  # ignore extras
            cx, cy, ww, hh = x*w, y*h, bw*w, bh*h
            x1, y1 = cx-ww/2, cy-hh/2
            ax.add_patch(patches.Rectangle((x1,y1), ww, hh, ec='lime', fc='none', lw=2))
            ax.text(x1, max(0, y1-3), names[int(c)], color='yellow',
                    fontsize=9, bbox=dict(facecolor='black', alpha=0.5))
    ax.axis('off'); plt.show()

pool = list(train_dir.glob("*.jpg")) + list(train_dir.glob("*.png")) + list(train_dir.glob("*.jpeg"))
if len(pool) >= 3:
    for p in random.sample(pool, k=3):
        show_one(p)
else:
    print("Not enough images to preview.")


## 4) Patch Ultralytics 8.3.32 Ray-Tune callback bug (no restart)

In [ ]:

# HARD DISABLE Ray Tune crash path
import types, importlib
try:
    ray = importlib.import_module("ray")
except Exception:
    ray = types.SimpleNamespace(); import sys; sys.modules["ray"] = ray
if not hasattr(ray, "train"): ray.train = types.SimpleNamespace()
if not hasattr(ray.train, "_internal"): ray.train._internal = types.SimpleNamespace()
ray.train._internal.session = types.SimpleNamespace(_get_session=lambda: None)
print("Patched: ray.train._internal.session._get_session -> None")

from ultralytics.utils import callbacks as _ulx_cb
for k, v in list(_ulx_cb.default_callbacks.items()):
    if isinstance(v, list):
        _ulx_cb.default_callbacks[k] = [cb for cb in v if "raytune" not in getattr(cb, "__module__", "")]
    elif callable(v) and "raytune" in getattr(v, "__module__", ""):
        _ulx_cb.default_callbacks.pop(k, None)
print("Scrubbed Ultralytics default callbacks of raytune ✅")

try:
    import ultralytics.utils.callbacks.raytune as _ulx_raytune
    def _noop(*args, **kwargs): ...
    _ulx_raytune.on_fit_epoch_end = _noop
    print("Patched ultralytics.utils.callbacks.raytune.on_fit_epoch_end → no-op")
except Exception as e:
    print("raytune module patch skipped:", e)


In [ ]:
# ==== KILL the Ray Tune callback in Ultralytics 8.3.32 (no restart) ====
import types, importlib, sys

# 1) Ensure a ray namespace exists and spoof the problematic function
try:
    ray = importlib.import_module("ray")
except Exception:
    ray = types.SimpleNamespace()
    sys.modules["ray"] = ray
if not hasattr(ray, "train"):
    ray.train = types.SimpleNamespace()
if not hasattr(ray.train, "_internal"):
    ray.train._internal = types.SimpleNamespace()
ray.train._internal.session = types.SimpleNamespace(_get_session=lambda: None)
print("Patched: ray.train._internal.session._get_session -> None")

# 2) Scrub Ultralytics default callbacks of any raytune hooks
from ultralytics.utils import callbacks as _ulx_cb
for k, v in list(_ulx_cb.default_callbacks.items()):
    if isinstance(v, list):
        _ulx_cb.default_callbacks[k] = [cb for cb in v if "raytune" not in getattr(cb, "__module__", "")]
    elif callable(v) and "raytune" in getattr(v, "__module__", ""):
        _ulx_cb.default_callbacks.pop(k, None)
print("Removed raytune hooks from Ultralytics default callbacks ✅")

# 3) (belt & suspenders) no-op the raytune module if it exists
try:
    import ultralytics.utils.callbacks.raytune as _ulx_raytune
    def _noop(*args, **kwargs): ...
    _ulx_raytune.on_fit_epoch_end = _noop
    print("Patched ultralytics.utils.callbacks.raytune.on_fit_epoch_end → no-op")
except Exception as e:
    print("raytune submodule not present or already inert:", e)


## 5) Train detector (YOLO11-n, T4-safe)

In [ ]:

gc.collect(); torch.cuda.empty_cache()

detector = YOLO("yolo11n.pt")
detector.callbacks = {}  # disable any leftover callbacks

detector.train(
    data=str(DATA_YAML),
    imgsz=512,
    epochs=30,
    batch=4,
    device=0,
    workers=1,
    optimizer="AdamW",
    amp=True,
    cos_lr=True,
    patience=5,
    cache=True,                # now /kaggle/working is writable
    mosaic=0.25, mixup=0.0,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    fliplr=0.5, degrees=5.0, scale=0.5, shear=2.0
)

det_val = detector.val(data=str(DATA_YAML), imgsz=512, split="val")
print(det_val.results_dict)


## 6) Save detector metrics to CSV

In [ ]:

dres = detector.val(data=str(DATA_YAML), imgsz=512, split="val")
pd.DataFrame(dres.results_dict.items(), columns=["metric","value"]).to_csv("/kaggle/working/detector_eval.csv", index=False)
print("Saved /kaggle/working/detector_eval.csv")


In [ ]:
import shutil, os
shutil.copytree("runs/detect/train", "/kaggle/working/detector_run", dirs_exist_ok=True)
print("Detector training saved ✅")


In [1]:
from ultralytics import YOLO
detector = YOLO("/kaggle/working/detector_run/weights/best.pt")  # load your trained model
results = detector.val(data="/kaggle/working/data.yaml", imgsz=512, split="val")
print(results.results_dict)


Ultralytics 8.3.32 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
YOLO11n summary (fused): 238 layers, 2,653,345 parameters, 0 gradients, 6.7 GFLOPs


val: Scanning /kaggle/working/e-waste/valid/labels.cache... 414 images, 1 backgrounds, 0 corrupt: 100%|██████████| 414/414 [00:00<?, ?it/s]

WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 90, len(boxes) = 559. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 26/26 [00:03<00:00,  6.90it/s]


                   all        414        559      0.273     0.0263     0.0201    0.00806
       Battery Charger          1          1          0          0          0          0
                  Bulb          7          7          1          0    0.00632    0.00505
          Coffee Maker          1          1          0          0          0          0
            DVD Player          2          2          0          0          0          0
              Decorder          2          2          0          0          0          0
           Dish washer          1          1          0          0          0          0
                   Fan          1          1          0          0          0          0
           Fax Machine          2          2          0          0          0          0
           Fly Swatter          1          1          0          0          0          0
                Fridge          1          1          0          0          0          0
                  HDP

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


Speed: 0.5ms preprocess, 3.2ms inference, 0.0ms loss, 2.0ms postprocess per image
Results saved to runs/detect/val
{'metrics/precision(B)': 0.27262189600502645, 'metrics/recall(B)': 0.026257861635220123, 'metrics/mAP50(B)': 0.02008465091315595, 'metrics/mAP50-95(B)': 0.00806295128727512, 'fitness': 0.009265121249863204}


## 7) Generate GT crops for classifier

In [ ]:

CLS_ROOT = Path("/kaggle/working/ewaste_cls")
(CLS_ROOT/"train").mkdir(parents=True, exist_ok=True)
(CLS_ROOT/"val").mkdir(parents=True, exist_ok=True)

def make_gt_crops(img_dir, out_dir):
    imgs = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png")) + list(img_dir.glob("*.jpeg"))
    for ip in imgs:
        im_bgr = cv2.imread(str(ip))
        if im_bgr is None: 
            continue
        im = cv2.cvtColor(im_bgr, cv2.COLOR_BGR2RGB)
        h, w = im.shape[:2]
        lp = Path(str(ip).replace("/images/","/labels/")).with_suffix(".txt")
        if not lp.exists(): 
            continue
        for ln in lp.read_text().splitlines():
            if not ln.strip(): continue
            vals = ln.split()
            if len(vals) < 5: continue
            cid, x, y, bw, bh = map(float, vals[:5])
            cx, cy, ww, hh = x*w, y*h, bw*w, bh*h
            x1, y1 = int(max(0, cx-ww/2)), int(max(0, cy-hh/2))
            x2, y2 = int(min(w-1, cx+ww/2)), int(min(h-1, cy+hh/2))
            if x2 <= x1 or y2 <= y1: continue
            crop = im[y1:y2, x1:x2]
            if crop.shape[0] < 16 or crop.shape[1] < 16: continue
            cname = names[int(cid)]
            outc = out_dir / cname; outc.mkdir(parents=True, exist_ok=True)
            cv2.imwrite(str(outc / f"{ip.stem}_{x1}_{y1}_{x2}_{y2}.jpg"),
                        cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))

make_gt_crops(train_dir, CLS_ROOT/"train")
make_gt_crops(val_dir,   CLS_ROOT/"val")
print("GT crops at:", CLS_ROOT)


## 8) Sync `val/` classes to match `train/` (no leakage)

In [ ]:

train_classes = sorted([d.name for d in (CLS_ROOT/'train').iterdir() if d.is_dir()])
val_classes   = sorted([d.name for d in (CLS_ROOT/'val').iterdir() if d.is_dir()])

missing_in_val = sorted(set(train_classes) - set(val_classes))
for c in missing_in_val:
    (CLS_ROOT/'val'/c).mkdir(parents=True, exist_ok=True)

print(f"Train classes: {len(train_classes)}, Val classes(after sync): {len([d.name for d in (CLS_ROOT/'val').iterdir() if d.is_dir()])}")


In [8]:
from ultralytics import YOLO
from pathlib import Path

# If you have partial progress, start from that; else from pretrained
last_ckpt = Path("runs/classify/train2/weights/last.pt")
start_weights = str(last_ckpt) if last_ckpt.exists() else "yolo11n-cls.pt"
print("Starting from:", start_weights)

clf = YOLO(start_weights)
clf.callbacks = {}  # extra safety

results = clf.train(
    data=str(CLS_ROOT),   # directory with train/ and val/
    imgsz=224,
    epochs=20,
    batch=64,             # drop to 32 if OOM
    workers=1,
    optimizer="AdamW",
    amp=True,
    cos_lr=True,
    project="runs/classify",
    name="train2_fix",    # new run name so it won't pull old args
    resume=False          # IMPORTANT: avoid Imagenet arg leak
)


Starting from: runs/classify/train2/weights/last.pt
New https://pypi.org/project/ultralytics/8.3.186 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.32 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=classify, mode=train, model=runs/classify/train2/weights/last.pt, data=/kaggle/working/ewaste_cls, epochs=20, time=None, patience=100, batch=64, imgsz=224, save=True, save_period=-1, cache=False, device=None, workers=1, project=runs/classify, name=train2_fix, exist_ok=False, pretrained=True, optimizer=AdamW, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=True, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=Fa

train: Scanning /kaggle/working/ewaste_cls/train... 10170 images, 0 corrupt: 100%|██████████| 10170/10170 [00:00<?, ?it/s]
val: Scanning /kaggle/working/ewaste_cls/val... 549 images, 0 corrupt: 100%|██████████| 549/549 [00:00<?, ?it/s]


optimizer: AdamW(lr=0.01, momentum=0.937) with parameter groups 39 weight(decay=0.0), 40 weight(decay=0.0005), 40 bias(decay=0.0)
TensorBoard: model graph visualization added ✅
Image sizes 224 train, 224 val
Using 1 dataloader workers
Logging results to runs/classify/train2_fix
Starting training for 20 epochs...

      Epoch    GPU_mem       loss  Instances       Size


               classes   top1_acc   top5_acc: 100%|██████████| 5/5 [00:01<00:00,  3.78it/s]

                   all      0.481      0.747


...

      20/20     0.904G     0.1877         58        224: 100%|██████████| 159/159 [01:02<00:00,  2.54it/s]
               classes   top1_acc   top5_acc: 100%|██████████| 5/5 [00:01<00:00,  3.80it/s]

                   all      0.725        0.9



20 epochs completed in 0.361 hours.
Optimizer stripped from runs/classify/train2_fix/weights/last.pt, 3.5MB
Optimizer stripped from runs/classify/train2_fix/weights/best.pt, 3.5MB

Validating runs/classify/train2_fix/weights/best.pt...
Ultralytics 8.3.32 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
YOLO11n-cls summary (fused): 112 layers, 1,669,496 parameters, 0 gradients, 3.3 GFLOPs
train: /kaggle/working/ewaste_cls/train... found 10170 images in 112 classes ✅ 
val: /kaggle/working/ewaste_cls/val... found 549 images in 48 classes: ERROR ❌️ requires 112 classes, not 48
test: None...


               classes   top1_acc   top5_acc: 100%|██████████| 5/5 [00:01<00:00,  3.37it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all      0.725      0.902
Speed: 0.1ms preprocess, 0.2ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to runs/classify/train2_fix


## 10) Save classifier metrics (+ confusion matrix)

In [10]:
import pandas as pd

cres = clf.val(data=str(CLS_ROOT), imgsz=224, split="val")
pd.DataFrame(cres.results_dict.items(), columns=["metric","value"]).to_csv("/kaggle/working/classifier_eval.csv", index=False)

# Confusion matrix (if available)
try:
    cres.plot_confusion_matrix(save_dir="runs/classify/confmat")
    if hasattr(cres, "confusion_matrix") and cres.confusion_matrix is not None:
        cm = cres.confusion_matrix.matrix
        labels = cres.confusion_matrix.labels
        pd.DataFrame(cm, index=labels, columns=labels).to_csv("/kaggle/working/classifier_confusion_matrix.csv")
except Exception as e:
    print("Confusion matrix export skipped:", e)

print("Saved /kaggle/working/classifier_eval.csv")


Ultralytics 8.3.32 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
train: /kaggle/working/ewaste_cls/train... found 10170 images in 112 classes ✅ 
val: /kaggle/working/ewaste_cls/val... found 549 images in 48 classes: ERROR ❌️ requires 112 classes, not 48
test: None...


val: Scanning /kaggle/working/ewaste_cls/val... 549 images, 0 corrupt: 100%|██████████| 549/549 [00:00<?, ?it/s]
               classes   top1_acc   top5_acc: 100%|██████████| 9/9 [00:02<00:00,  3.80it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all      0.727      0.903
Speed: 0.1ms preprocess, 0.3ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to runs/classify/train2_fix3
Confusion matrix export skipped: 'ClassifyMetrics' object has no attribute 'plot_confusion_matrix'. See valid attributes below.

    Class for computing classification metrics including top-1 and top-5 accuracy.

    Attributes:
        top1 (float): The top-1 accuracy.
        top5 (float): The top-5 accuracy.
        speed (Dict[str, float]): A dictionary containing the time taken for each step in the pipeline.
        fitness (float): The fitness of the model, which is equal to top-5 accuracy.
        results_dict (Dict[str, Union[float, str]]): A dictionary containing the classification metrics and fitness.
        keys (List[str]): A list of keys for the results_dict.

    Methods:
        process(targets, pred): Processes the targets and predictions to compute classification metrics.
    
Saved /kaggle/working/cl

## 11) One-cell summary report

In [11]:

import json

det = pd.read_csv("/kaggle/working/detector_eval.csv").set_index("metric")["value"].to_dict() if Path("/kaggle/working/detector_eval.csv").exists() else {}
clf_m = pd.read_csv("/kaggle/working/classifier_eval.csv").set_index("metric")["value"].to_dict() if Path("/kaggle/working/classifier_eval.csv").exists() else {}

summary = {
    "detection": {
        "precision": det.get("metrics/precision(B)"),
        "recall": det.get("metrics/recall(B)"),
        "mAP@0.5": det.get("metrics/mAP50(B)"),
        "mAP@0.5:0.95": det.get("metrics/mAP50-95(B)")
    },
    "classification": {
        "top1_acc": clf_m.get("metrics/accuracy_top1"),
        "top5_acc": clf_m.get("metrics/accuracy_top5")
    }
}
print(json.dumps(summary, indent=2))


{
  "detection": {
    "precision": 0.6294418547346533,
    "recall": 0.1799766001057856,
    "mAP@0.5": 0.2054976412778637,
    "mAP@0.5:0.95": 0.1469800024768481
  },
  "classification": {
    "top1_acc": 0.7267759442329407,
    "top5_acc": 0.9034608602523804
  }
}


## 12) (Optional) End-to-end inference: annotate detections + classifier labels

In [14]:
from pathlib import Path
from PIL import Image, ImageDraw
import cv2

# --- Define dataset roots ---
DST = Path("/kaggle/working/e-waste")      # detection dataset root
CLS_ROOT = Path("/kaggle/working/ewaste_cls")  # classification dataset root

OUT_DIR = Path("/kaggle/working/preds")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Choose test set if exists, else fallback to validation
infer_dir = DST / "test/images"
if not infer_dir.exists() or len(list(infer_dir.glob("*"))) == 0:
    infer_dir = DST / "valid/images"

# Collect classifier class names
cls_names = [p.name for p in sorted((CLS_ROOT/"train").iterdir()) if p.is_dir()]

def draw_box_and_label(pil_img, box, text):
    """Draws YOLO bbox + classification label on image."""
    x1, y1, x2, y2 = map(int, box)
    draw = ImageDraw.Draw(pil_img)
    draw.rectangle([x1, y1, x2, y2], outline=(0, 255, 0), width=3)
    tw = draw.textlength(text); th = 14
    draw.rectangle([x1, max(0, y1 - 18), x1 + int(tw) + 8, y1], fill=(0, 0, 0))
    draw.text((x1 + 4, y1 - 16), text, fill=(255, 255, 0))
    return pil_img

# --- Run inference ---
names = detector.model.names   # detection class labels

for r in detector.predict(source=str(infer_dir), imgsz=512, conf=0.25, stream=True, verbose=False):
    im_bgr = r.orig_img
    h, w = im_bgr.shape[:2]
    pil_im = Image.fromarray(cv2.cvtColor(im_bgr, cv2.COLOR_BGR2RGB))

    for b in r.boxes:
        x1, y1, x2, y2 = b.xyxy[0].cpu().numpy()
        x1, y1 = max(0, int(x1)), max(0, int(y1))
        x2, y2 = min(w - 1, int(x2)), min(h - 1, int(y2))
        if x2 <= x1 or y2 <= y1:
            continue

        # Crop detection region
        crop = im_bgr[y1:y2, x1:x2]
        if crop.shape[0] < 16 or crop.shape[1] < 16:
            continue

        # Run classifier on crop
        cres = clf.predict(crop, imgsz=224, verbose=False)
        cprobs = cres[0].probs
        top1 = int(cprobs.top1)
        conf = float(cprobs.top1conf)
        cls_label = cls_names[top1] if top1 < len(cls_names) else f"id{top1}"

        # Combine detector + classifier label
        det_cid = int(b.cls.item())
        label = f"{names[det_cid]} → {cls_label} ({conf:.2f})"
        pil_im = draw_box_and_label(pil_im, (x1, y1, x2, y2), label)

    # Save annotated image
    out_path = OUT_DIR / f"{Path(r.path).stem}_pred.jpg"
    pil_im.save(out_path)

print("✅ Annotated images saved in:", OUT_DIR)


✅ Annotated images saved in: /kaggle/working/preds


In [21]:
# Detector best weights (replace "train" with your actual detector run name if different)
!cp runs/detect/train/weights/best.pt /kaggle/working/detector_best.pt

# Classifier best weights (pick the latest stable one, here train2_fix)
!cp runs/classify/train2_fix/weights/best.pt /kaggle/working/classifier_best.pt

# Just to double-check they’re copied
!ls -lh /kaggle/working/*best.pt


-rw-r--r-- 1 root root 3.4M Aug 26 22:59 /kaggle/working/classifier_best.pt
-rw-r--r-- 1 root root  16M Aug 26 22:59 /kaggle/working/detector_best.pt
